In [ ]:
import pandas as pd
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("Ecommerce_BigData_Project") \
    .getOrCreate()

In [ ]:
print(spark)

In [ ]:
df = spark.read.csv(
    "/content/SampleSuperstore.csv",
    header=True,
    inferSchema=True
)

In [ ]:
df.show(5)

+--------------+---------+-------------+---------------+----------+-----------+------+---------------+------------+--------+--------+--------+--------+
|     Ship Mode|  Segment|      Country|           City|     State|Postal Code|Region|       Category|Sub-Category|   Sales|Quantity|Discount|  Profit|
+--------------+---------+-------------+---------------+----------+-----------+------+---------------+------------+--------+--------+--------+--------+
|  Second Class| Consumer|United States|      Henderson|  Kentucky|      42420| South|      Furniture|   Bookcases|  261.96|       2|     0.0| 41.9136|
|  Second Class| Consumer|United States|      Henderson|  Kentucky|      42420| South|      Furniture|      Chairs|  731.94|       3|     0.0| 219.582|
|  Second Class|Corporate|United States|    Los Angeles|California|      90036|  West|Office Supplies|      Labels|   14.62|       2|     0.0|  6.8714|
|Standard Class| Consumer|United States|Fort Lauderdale|   Florida|      33311| South|  

In [ ]:
df.printSchema()

root
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [ ]:
df.count()

9994

In [ ]:
print(df.columns)

['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']


In [ ]:
from pyspark.sql.functions import col, when, count

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

+---------+-------+-------+----+-----+-----------+------+--------+------------+-----+--------+--------+------+
|Ship Mode|Segment|Country|City|State|Postal Code|Region|Category|Sub-Category|Sales|Quantity|Discount|Profit|
+---------+-------+-------+----+-----+-----------+------+--------+------------+-----+--------+--------+------+
|        0|      0|      0|   0|    0|          0|     0|       0|           0|    0|       0|       0|     0|
+---------+-------+-------+----+-----+-----------+------+--------+------------+-----+--------+--------+------+



In [ ]:
total_rows = df.count()

unique_rows = df.dropDuplicates().count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", total_rows - unique_rows)

Total Rows: 9994
Unique Rows: 9977
Duplicate Rows: 17


In [ ]:
df.describe().show()

+-------+--------------+-----------+-------------+--------+-------+------------------+-------+----------+------------+-----------------+-----------------+-------------------+------------------+
|summary|     Ship Mode|    Segment|      Country|    City|  State|       Postal Code| Region|  Category|Sub-Category|            Sales|         Quantity|           Discount|            Profit|
+-------+--------------+-----------+-------------+--------+-------+------------------+-------+----------+------------+-----------------+-----------------+-------------------+------------------+
|  count|          9994|       9994|         9994|    9994|   9994|              9994|   9994|      9994|        9994|             9994|             9994|               9994|              9994|
|   mean|          NULL|       NULL|         NULL|    NULL|   NULL|  55190.3794276566|   NULL|      NULL|        NULL|229.8580008304938|3.789573744246548|0.15620272163298934|28.656896307784802|
| stddev|          NULL|      

In [ ]:
loss_df = df.filter(
    col("Profit") < 0
)

loss_df.show(5)

+--------------+-----------+-------------+---------------+------------+-----------+-------+---------------+------------+--------+--------+--------+----------+
|     Ship Mode|    Segment|      Country|           City|       State|Postal Code| Region|       Category|Sub-Category|   Sales|Quantity|Discount|    Profit|
+--------------+-----------+-------------+---------------+------------+-----------+-------+---------------+------------+--------+--------+--------+----------+
|Standard Class|   Consumer|United States|Fort Lauderdale|     Florida|      33311|  South|      Furniture|      Tables|957.5775|       5|    0.45|  -383.031|
|Standard Class|Home Office|United States|     Fort Worth|       Texas|      76106|Central|Office Supplies|  Appliances|   68.81|       5|     0.8|  -123.858|
|Standard Class|Home Office|United States|     Fort Worth|       Texas|      76106|Central|Office Supplies|     Binders|   2.544|       3|     0.8|    -3.816|
|  Second Class|   Consumer|United States|   P

In [ ]:
df_clean = df.dropDuplicates()

In [ ]:
df_clean.count()

9977

In [ ]:
from pyspark.sql.functions import sum
from pyspark.sql.functions import round

df_clean.select(
    round(sum("Sales"),2).alias("Total_Sales")
).show()

+-----------+
|Total_Sales|
+-----------+
| 2296195.59|
+-----------+



In [ ]:
df_clean.select(
    round(sum("Profit"),2).alias("Total_Profit")
).show()

+------------+
|Total_Profit|
+------------+
|   286241.42|
+------------+



In [ ]:
df_clean.groupby("category")\
.agg(sum("sales").alias("Total_Sales"))\
.orderBy(sum("sales"), ascending=False)\
.show()


+---------------+-----------------+
|       category|      Total_Sales|
+---------------+-----------------+
|     Technology|836154.0329999969|
|      Furniture|741306.3132999991|
|Office Supplies|718735.2439999988|
+---------------+-----------------+



In [ ]:
df_clean.groupBy("Category") \
    .agg(sum("Profit").alias("Total_Profit")) \
    .orderBy("Total_Profit", ascending=False) \
    .show()

+---------------+------------------+
|       Category|      Total_Profit|
+---------------+------------------+
|     Technology| 145454.9480999996|
|Office Supplies|122364.66080000032|
|      Furniture|18421.813699999984|
+---------------+------------------+



In [ ]:


df_clean.groupBy("Sub-Category") \
    .agg(sum("Profit").alias("Total_Profit")) \
    .orderBy("Total_Profit", ascending=False) \
    .show(10)

+------------+------------------+
|Sub-Category|      Total_Profit|
+------------+------------------+
|     Copiers| 55617.82490000002|
|      Phones| 44515.73060000007|
| Accessories| 41936.63569999996|
|       Paper| 33944.23950000002|
|     Binders| 30228.00030000002|
|      Chairs| 26567.12780000002|
|     Storage|21278.826400000005|
|  Appliances|18138.005399999976|
| Furnishings|13052.723000000007|
|   Envelopes| 6964.176699999998|
+------------+------------------+
only showing top 10 rows


In [ ]:
df_clean.groupBy("Sub-Category") \
    .agg(sum("Sales").alias("Total_Sales")) \
    .orderBy("Total_Sales", ascending=False) \
    .show(10)

+------------+------------------+
|Sub-Category|       Total_Sales|
+------------+------------------+
|      Phones|330007.05399999995|
|      Chairs|327777.76100000046|
|     Storage|223843.60800000018|
|      Tables| 206965.5320000002|
|     Binders|203409.16899999982|
|    Machines|189238.63100000005|
| Accessories| 167380.3179999998|
|     Copiers|         149528.03|
|   Bookcases|114879.99629999996|
|  Appliances| 107532.1610000001|
+------------+------------------+
only showing top 10 rows


In [ ]:
df_clean.groupBy("Region") \
    .agg(sum("Sales").alias("Total_Sales")) \
    .orderBy("Total_Sales", ascending=False) \
    .show()

+-------+------------------+
| Region|       Total_Sales|
+-------+------------------+
|   West| 725255.6365000005|
|   East| 678435.1959999991|
|Central| 500782.8528000008|
|  South|391721.90500000044|
+-------+------------------+



In [ ]:
df_clean.groupBy("Region") \
    .agg(sum("Profit").alias("Total_Profit")) \
    .orderBy("Total_Profit", ascending=False) \
    .show()

+-------+------------------+
| Region|      Total_Profit|
+-------+------------------+
|   West|108329.80790000015|
|   East| 91506.30920000008|
|  South| 46749.43030000003|
|Central|39655.875200000126|
+-------+------------------+



In [ ]:


df_clean.groupBy("Sub-Category") \
    .agg(sum("Profit").alias("Total_Profit")) \
    .orderBy("Total_Profit", ascending=False) \
    .show(10, truncate=False)

+------------+------------------+
|Sub-Category|Total_Profit      |
+------------+------------------+
|Copiers     |55617.82490000002 |
|Phones      |44515.73060000007 |
|Accessories |41936.63569999996 |
|Paper       |33944.23950000002 |
|Binders     |30228.00030000002 |
|Chairs      |26567.12780000002 |
|Storage     |21278.826400000005|
|Appliances  |18138.005399999976|
|Furnishings |13052.723000000007|
|Envelopes   |6964.176699999998 |
+------------+------------------+
only showing top 10 rows


In [ ]:
df_clean.groupBy("Sub-Category") \
    .agg(sum("Sales").alias("Total_Sales")) \
    .orderBy("Total_Sales", ascending=False) \
    .show(10, truncate=False)

+------------+------------------+
|Sub-Category|Total_Sales       |
+------------+------------------+
|Phones      |330007.05399999995|
|Chairs      |327777.76100000046|
|Storage     |223843.60800000018|
|Tables      |206965.5320000002 |
|Binders     |203409.16899999982|
|Machines    |189238.63100000005|
|Accessories |167380.3179999998 |
|Copiers     |149528.03         |
|Bookcases   |114879.99629999996|
|Appliances  |107532.1610000001 |
+------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import avg

df_clean.groupBy("Region") \
    .agg(
        avg("Discount").alias("Avg_Discount"),
        avg("Profit").alias("Avg_Profit")
    ) \
    .orderBy("Avg_Discount", ascending=False) \
    .show()

+-------+-------------------+------------------+
| Region|       Avg_Discount|        Avg_Profit|
+-------+-------------------+------------------+
|Central| 0.2402501078050885|17.100420526088886|
|  South|0.14725308641975088|28.857673024691376|
|   East|0.14534270650263353|32.163904815465756|
|   West|0.10961478233635796| 33.92728089570941|
+-------+-------------------+------------------+



In [ ]:

df_clean.groupBy("Segment") \
    .agg(
        sum("Sales").alias("Total_Sales"),
        sum("Profit").alias("Total_Profit")
    ) \
    .orderBy("Total_Sales", ascending=False) \
    .show()

+-----------+------------------+------------------+
|    Segment|       Total_Sales|      Total_Profit|
+-----------+------------------+------------------+
|   Consumer|1160832.7749999908|134007.44129999986|
|  Corporate| 706070.1307999996| 91954.97980000002|
|Home Office| 429292.6845000001|60279.001500000166|
+-----------+------------------+------------------+



In [ ]:
from pyspark.sql.functions import sum

df_clean.select(
    sum("Sales").alias("Total_Sales")
).show()

+-----------------+
|      Total_Sales|
+-----------------+
|2296195.590299954|
+-----------------+



In [ ]:
df_clean.select(
    sum("Profit").alias("Total_Profit")
).show()

+------------------+
|      Total_Profit|
+------------------+
|286241.42260000017|
+------------------+



In [ ]:

df_clean.select(
    sum("Quantity").alias("Total_Quantity")
).show()

+--------------+
|Total_Quantity|
+--------------+
|         37820|
+--------------+



In [ ]:

df_clean.select(
    avg("Discount").alias("Average_Discount")
).show()

+-------------------+
|   Average_Discount|
+-------------------+
|0.15627844041295968|
+-------------------+



In [ ]:
totals = df_clean.select(
    sum("Sales").alias("sales"),
    sum("Profit").alias("profit")
).collect()[0]

profit_margin = (
    totals["profit"] /
    totals["sales"]
) * 100

print("Profit Margin (%):", __builtins__.round(profit_margin, 2))

Profit Margin (%): 12.47


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, sum

subcategory_profit = (
    df_clean.groupBy("Sub-Category")
    .agg(sum("Profit").alias("Total_Profit"))
)

window_spec = Window.orderBy(
    subcategory_profit["Total_Profit"].desc()
)

ranked_df = (
    subcategory_profit
    .withColumn(
        "Profit_Rank",
        rank().over(window_spec)
    )
)

ranked_df.show()

+------------+-------------------+-----------+
|Sub-Category|       Total_Profit|Profit_Rank|
+------------+-------------------+-----------+
|     Copiers|  55617.82490000002|          1|
|      Phones|  44515.73060000007|          2|
| Accessories|  41936.63569999996|          3|
|       Paper|  33944.23950000002|          4|
|     Binders|  30228.00030000002|          5|
|      Chairs|  26567.12780000002|          6|
|     Storage| 21278.826400000005|          7|
|  Appliances| 18138.005399999976|          8|
| Furnishings| 13052.723000000007|          9|
|   Envelopes|  6964.176699999998|         10|
|         Art|          6524.6118|         11|
|      Labels| 5526.3820000000005|         12|
|    Machines|  3384.756900000001|         13|
|   Fasteners|  949.5182000000001|         14|
|    Supplies|-1189.0994999999984|         15|
|   Bookcases| -3472.555999999999|         16|
|      Tables|        -17725.4811|         17|
+------------+-------------------+-----------+



In [ ]:


region_profit = (
    df_clean.groupBy("Region")
    .agg(sum("Profit").alias("Total_Profit"))
)

window_spec = Window.orderBy(
    region_profit["Total_Profit"].desc()
)

ranked_region = (
    region_profit
    .withColumn(
        "Profit_Rank",
        rank().over(window_spec)
    )
)

ranked_region.show()

+-------+------------------+-----------+
| Region|      Total_Profit|Profit_Rank|
+-------+------------------+-----------+
|   West|108329.80790000015|          1|
|   East| 91506.30920000008|          2|
|  South| 46749.43030000003|          3|
|Central|39655.875200000126|          4|
+-------+------------------+-----------+



In [ ]:
print(
    "Partitions:",
    df_clean.rdd.getNumPartitions()
)

Partitions: 1


In [ ]:
df_repartitioned = df_clean.repartition(4)

print(
    "Partitions:",
    df_repartitioned.rdd.getNumPartitions()
)

Partitions: 4


In [ ]:
df.coalesce(1)

DataFrame[Ship Mode: string, Segment: string, Country: string, City: string, State: string, Postal Code: int, Region: string, Category: string, Sub-Category: string, Sales: double, Quantity: int, Discount: double, Profit: double]

In [ ]:
print(
    "Partitions:",
    df_clean.rdd.getNumPartitions()
)

Partitions: 1


In [ ]:
df_clean.write.mode("overwrite") \
    .option("header", True) \
    .csv("/content/clean_superstore")

In [ ]:
from pyspark.sql.functions import sum

category_summary = (
    df_clean.groupBy("Category")
    .agg(
        sum("Sales").alias("Total_Sales"),
        sum("Profit").alias("Total_Profit")
    )
)

category_summary.show()

+---------------+-----------------+------------------+
|       Category|      Total_Sales|      Total_Profit|
+---------------+-----------------+------------------+
|Office Supplies|718735.2439999988|122364.66080000032|
|      Furniture|741306.3132999991|18421.813699999984|
|     Technology|836154.0329999969| 145454.9480999996|
+---------------+-----------------+------------------+



In [ ]:
region_summary = (
    df_clean.groupBy("Region")
    .agg(
        sum("Sales").alias("Total_Sales"),
        sum("Profit").alias("Total_Profit")
    )
)

region_summary.show()

+-------+------------------+------------------+
| Region|       Total_Sales|      Total_Profit|
+-------+------------------+------------------+
|  South|391721.90500000044| 46749.43030000003|
|Central| 500782.8528000008|39655.875200000126|
|   East| 678435.1959999991| 91506.30920000008|
|   West| 725255.6365000005|108329.80790000015|
+-------+------------------+------------------+



In [ ]:
segment_summary = (
    df_clean.groupBy("Segment")
    .agg(
        sum("Sales").alias("Total_Sales"),
        sum("Profit").alias("Total_Profit")
    )
)

segment_summary.show()

+-----------+------------------+------------------+
|    Segment|       Total_Sales|      Total_Profit|
+-----------+------------------+------------------+
|   Consumer|1160832.7749999908|134007.44129999986|
|Home Office| 429292.6845000001|60279.001500000166|
|  Corporate| 706070.1307999996| 91954.97980000002|
+-----------+------------------+------------------+



In [ ]:
category_summary.coalesce(1) \
.write.mode("overwrite") \
.option("header", True) \
.csv("/content/category_summary")

In [ ]:
region_summary.coalesce(1) \
.write.mode("overwrite") \
.option("header", True) \
.csv("/content/region_summary")

In [ ]:
segment_summary.coalesce(1) \
.write.mode("overwrite") \
.option("header", True) \
.csv("/content/segment_summary")